# Monolingual RAG — German only (FDV DE)

Indexes **only** the German documents (`data/de/`).  
French and Italian queries are handled cross-lingually via the shared multilingual E5 embedding space.

## Section 0 — Installation

> **Before running this notebook**, pull the Ollama model once in a terminal:
> ```
> ollama pull phi4-mini
> ```
> Install PyTorch (CPU-only) separately if needed:
> ```
> uv pip install torch --index-url https://download.pytorch.org/whl/cpu
> ```

In [ ]:
%pip install -q sentence-transformers chromadb rank-bm25 langdetect ollama

## Section 1 — Imports & Configuration

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import chromadb
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder

from chunking    import load_documents, build_chunk_records
from retrieval   import build_bm25_index, retrieve
from generation  import ask

# ── Configuration ─────────────────────────────────────────────────────────
DATA_DIRS       = {"de": Path("data/de")}
COLLECTION_NAME = "fdv_de_only"
CHROMA_DIR      = Path("chroma_db_de")
MODEL_NAME      = "phi4-mini"
EMBED_MODEL     = "intfloat/multilingual-e5-small"
CE_MODEL        = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
BATCH_SIZE      = 64    # embedding batch size

CFG = {
    "TOP_K":          3,
    "BM25_WEIGHT":    0.4,
    "SEMANTIC_POOL":  30,   # TOP_K * 10
    "RERANK_POOL":    6,    # TOP_K * 2
    "MODEL_NAME":     MODEL_NAME,
}

# Edit these to test retrieval in Section 6
TEST_QUERIES = [
    "Unter welchen Bedingungen sind Arbeiten ohne Sicherheitswärter zugelassen?",
    "Wann ist der Sicherheits-Zwischenraum vorhanden?",
    "Wann sind Transporte von Waren aller Art im bedienten Führerstand zugelassen?",
]

# Edit this to test Q&A in Section 7
QUERY = "Wann muss der Lokführer die Wirkung der Luftbremse prüfen?"

print("Configuration ready.")

Configuration ready.


## Section 2 — Document Loading & Chunking

In [2]:
all_docs = []
for lang, data_dir in DATA_DIRS.items():
    docs = load_documents(data_dir, lang)
    all_docs.extend(docs)
    print(f"  {lang}: {len(docs)} documents loaded")

chunk_records = build_chunk_records(all_docs)
print(f"\nTotal chunks: {len(chunk_records)}")

# Preview
preview = pd.DataFrame([
    {k: v for k, v in r.items() if k != "text"}
    for r in chunk_records[:5]
])
display(preview)

  de: 15 documents loaded

Total chunks: 4851


,source_file,document_title,regulation_number,section_id,section_title,chunk_index,language
0,01_Grundlagen.txt,Grundlagen,R 300.1,NaN,NaN,0,de
1,01_Grundlagen.txt,Grundlagen,R 300.1,1.2,Geltungsbereich,1,de
2,01_Grundlagen.txt,Grundlagen,R 300.1,1.2.1,Anwendbarkeit der Vorgaben nach Teil-Geltungsb...,2,de
3,01_Grundlagen.txt,Grundlagen,R 300.1,1.2.2,Anwendbarkeit der Vorgaben nach Funktionen,3,de
4,01_Grundlagen.txt,Grundlagen,R 300.1,1.2.3,Auswirkungen des europäischen Rechts,4,de


## Section 3 — Models

In [3]:
print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)

print("Loading cross-encoder...")
cross_encoder = CrossEncoder(CE_MODEL)

print("\nModel demo — cross-lingual similarity:")
demo_passages = [
    "passage: Bremsprobe bei Zügen",
    "passage: Freifahrt und Zugvorbereitung",
    "passage: Freins — essai de frein avant le départ",
]
demo_query = "query: Comment effectuer la vérification des freins?"

demo_embs = embedder.encode(demo_passages + [demo_query], normalize_embeddings=True)
q_emb     = demo_embs[-1]
sims      = demo_embs[:-1] @ q_emb

for passage, sim in zip(demo_passages, sims):
    print(f"  sim={sim:.3f}  {passage[:60]}")

Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5840.67it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading cross-encoder...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3933.32it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model demo — cross-lingual similarity:
  sim=0.820  passage: Bremsprobe bei Zügen
  sim=0.800  passage: Freifahrt und Zugvorbereitung
  sim=0.842  passage: Freins — essai de frein avant le départ


## Section 4 — Indexing

In [4]:
client     = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

if collection.count() > 0:
    print(f"Collection already contains {collection.count()} chunks — skipping re-indexing.")
else:
    print(f"Embedding and indexing {len(chunk_records)} chunks...")
    texts_to_embed = ["passage: " + r["text"] for r in chunk_records]

    all_embeddings = []
    for i in tqdm(range(0, len(texts_to_embed), BATCH_SIZE), desc="Embedding"):
        batch = texts_to_embed[i : i + BATCH_SIZE]
        embs  = embedder.encode(batch, normalize_embeddings=True, show_progress_bar=False)
        all_embeddings.extend(embs.tolist())

    # Add to ChromaDB in batches (None metadata values → "")
    CHROMA_BATCH = 5000
    for i in tqdm(range(0, len(chunk_records), CHROMA_BATCH), desc="Indexing"):
        batch_records = chunk_records[i : i + CHROMA_BATCH]
        batch_embs    = all_embeddings[i : i + CHROMA_BATCH]
        metadatas = [
            {k: (v if v is not None else "") for k, v in r.items() if k != "text"}
            for r in batch_records
        ]
        collection.add(
            ids=[str(i + j) for j in range(len(batch_records))],
            embeddings=batch_embs,
            documents=[r["text"] for r in batch_records],
            metadatas=metadatas,
        )

    print(f"Indexed {collection.count()} chunks.")

Embedding and indexing 4851 chunks...


Indexing: 100%|██████████| 1/1 [00:04<00:00,  4.70s/it]

Indexed 4851 chunks.


## Section 5 — BM25 Index

In [5]:
print("Building BM25 index...")
bm25_index = build_bm25_index(chunk_records)
print(f"BM25 index ready ({len(chunk_records)} documents).")

Building BM25 index...
BM25 index ready (4851 documents).


## Section 6 — Retrieval Test

Edit `TEST_QUERIES` in Section 1 to try different questions.

In [8]:
for query in TEST_QUERIES:
    print(f"\n{'─'*70}")
    print(f"Query: {query}")
    print(f"{'─'*70}")
    results = retrieve(query, embedder, cross_encoder,
                       collection, bm25_index, chunk_records, CFG)
    for j, r in enumerate(results, 1):
        sec = r.get("section_id") or ""
        print(f"  [{j}] {r.get('regulation_number','')} §{sec}  "
              f"rerank={r['rerank_score']:.3f}  "
              f"sem={r['sem_score']:.3f}  "
              f"bm25={r['bm25_score']:.3f}")
        print(f"       {r['text'][:120].strip()}...")


──────────────────────────────────────────────────────────────────────
Query: Unter welchen Bedingungen sind Arbeiten ohne Sicherheitswärter zugelassen?
──────────────────────────────────────────────────────────────────────
  [1] R 300.12 §3.1.6  rerank=4.049  sem=0.878  bm25=1.000
       3.1.6	Arbeiten ohne SIWÄ
	
	Arbeiten ohne SIWÄ sind nur zugelassen 
–	bei Arbeiten mit maximal 2 Personen, welche eine u...
  [2] R 300.12 §3.1.6  rerank=-0.436  sem=0.836  bm25=0.491
       selbst verantwortlich. Dies beinhaltet auch die vorausgehende Planung der eigenen Schutzmassnahmen. 
Als Anforderung für...
  [3] R 300.12 §5.3.3  rerank=-0.570  sem=0.863  bm25=0.910
       5.3.3	Arbeiten ohne Sicherheitsmassnahmen	
		
	Für Arbeiten ohne Sicherheitsmassnahmen ist keine weitere Planung der Sic...

──────────────────────────────────────────────────────────────────────
Query: Wann ist der Sicherheits-Zwischenraum vorhanden?
──────────────────────────────────────────────────────────────────────
  [1

## Section 7 — Single Query Answer

Edit `QUERY` in Section 1 to ask a different question.

In [9]:
print(f"Query: {QUERY}\n")
answer, chunks = ask(QUERY, embedder, cross_encoder,
                     collection, bm25_index, chunk_records, CFG)

print("Answer:")
print(answer)

print("\nSources used:")
for r in chunks:
    sec  = r.get("section_id")    or ""
    ttl  = r.get("section_title") or ""
    lang = r.get("language")      or ""
    print(f"  [{lang.upper()}] {r.get('regulation_number','')} §{sec} — {ttl}")

Query: Wann muss der Lokführer die Wirkung der Luftbremse prüfen?

Answer:
Der Lokführer hat die Wirkung der Luftbremse unmittelbar nach dem Ingangsetzen des Triebfahrzeugs zu prüfen (Excerpt 3, Section 2.2.5). Außerdem muss er sie vor der Einfahrt ins starkes Gefälle oder in einen Kopfbahnhof überprüfen und vorrangig bei Temperaturen unter 0°C sofort nach Beginn der Fahrt tun (Excerpt 1, Section 2.3.7; Excerpt 3, Section 2.2.5).

Sources used:
  [DE] R 300.14 §2.3.7 — Bremsprobe auf Wirkung bei Zügen
  [DE] R 300.14 §2.3.5 — Bremsprobe bei Triebfahrzeugen
  [DE] R 300.14 §2.2.5 — Winterbetrieb
